# Most Viewed YouTube Videos - Full EDA
Exploring the top 1000 most viewed YouTube videos of all time

## 1. Data Loading & Overview

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re, glob, warnings

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

csv_path = glob.glob('/kaggle/input/**/*.csv', recursive=True)[0]
df = pd.read_csv(csv_path)

def parse_count(s):
    s = str(s).strip()
    if s.endswith('B'): return float(s[:-1]) * 1e9
    if s.endswith('M'): return float(s[:-1]) * 1e6
    if s.endswith('K'): return float(s[:-1]) * 1e3
    return float(s)

df['views_num'] = df['views'].apply(parse_count)
df['likes_num'] = df['likes'].apply(parse_count)
df['like_view_ratio'] = df['likes_num'] / df['views_num'] * 100
df['title'] = df['title'].apply(lambda x: re.sub(r'[^\x00-\x7F]+', '', str(x)).strip() or '???')
df.head()

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nDtypes:')
print(df.dtypes)
print(f'\nNulls:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

In [ ]:
df.describe()

## 2. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Views
axes[0,0].hist(df['views_num'] / 1e9, bins=50, color='#3b82f6', edgecolor='#1a1a2e', alpha=0.9)
axes[0,0].axvline(df['views_num'].median() / 1e9, color='#f97316', linestyle='--', label=f'Median: {df["views_num"].median()/1e9:.2f}B')
axes[0,0].axvline(df['views_num'].mean() / 1e9, color='#10b981', linestyle='--', label=f'Mean: {df["views_num"].mean()/1e9:.2f}B')
axes[0,0].set_xlabel('Views (Billions)')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Views Distribution', fontweight='bold', color='white')
axes[0,0].legend(fontsize=8)

# Likes
axes[0,1].hist(df['likes_num'] / 1e6, bins=50, color='#f97316', edgecolor='#1a1a2e', alpha=0.9)
axes[0,1].axvline(df['likes_num'].median() / 1e6, color='#3b82f6', linestyle='--', label=f'Median: {df["likes_num"].median()/1e6:.1f}M')
axes[0,1].axvline(df['likes_num'].mean() / 1e6, color='#10b981', linestyle='--', label=f'Mean: {df["likes_num"].mean()/1e6:.1f}M')
axes[0,1].set_xlabel('Likes (Millions)')
axes[0,1].set_ylabel('Count')
axes[0,1].set_title('Likes Distribution', fontweight='bold', color='white')
axes[0,1].legend(fontsize=8)

# Title length
axes[1,0].hist(df['title_length'], bins=40, color='#a855f7', edgecolor='#1a1a2e', alpha=0.9)
axes[1,0].axvline(df['title_length'].median(), color='#f97316', linestyle='--', label=f'Median: {df["title_length"].median():.0f}')
axes[1,0].set_xlabel('Title Length (chars)')
axes[1,0].set_ylabel('Count')
axes[1,0].set_title('Title Length Distribution', fontweight='bold', color='white')
axes[1,0].legend(fontsize=8)

# Like-View ratio
axes[1,1].hist(df['like_view_ratio'], bins=40, color='#10b981', edgecolor='#1a1a2e', alpha=0.9)
axes[1,1].axvline(df['like_view_ratio'].median(), color='#f97316', linestyle='--', label=f'Median: {df["like_view_ratio"].median():.2f}%')
axes[1,1].set_xlabel('Like/View Ratio (%)')
axes[1,1].set_ylabel('Count')
axes[1,1].set_title('Engagement Ratio Distribution', fontweight='bold', color='white')
axes[1,1].legend(fontsize=8)

for row in axes:
    for ax in row:
        ax.spines[['top', 'right']].set_visible(False)
        ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

In [ ]:
# Log-scale box plots
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

bp1 = axes[0].boxplot(df['views_num'] / 1e9, patch_artist=True, boxprops=dict(facecolor='#3b82f6'),
                       medianprops=dict(color='white'), flierprops=dict(marker='o', markersize=3, alpha=0.4))
axes[0].set_ylabel('Views (Billions)')
axes[0].set_title('Views', fontweight='bold', color='white')

bp2 = axes[1].boxplot(df['likes_num'] / 1e6, patch_artist=True, boxprops=dict(facecolor='#f97316'),
                       medianprops=dict(color='white'), flierprops=dict(marker='o', markersize=3, alpha=0.4))
axes[1].set_ylabel('Likes (Millions)')
axes[1].set_title('Likes', fontweight='bold', color='white')

bp3 = axes[2].boxplot(df['title_length'], patch_artist=True, boxprops=dict(facecolor='#a855f7'),
                       medianprops=dict(color='white'), flierprops=dict(marker='o', markersize=3, alpha=0.4))
axes[2].set_ylabel('Characters')
axes[2].set_title('Title Length', fontweight='bold', color='white')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 3. Categorical Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Content type
ct = df['content_type'].value_counts()
colors_ct = plt.cm.Set2(np.linspace(0, 1, len(ct)))
axes[0].barh(ct.index[::-1], ct.values[::-1], color=colors_ct[::-1], edgecolor='none')
for i, v in enumerate(ct.values[::-1]):
    axes[0].text(v + 2, i, str(v), va='center', fontsize=9, color='#ccc')
axes[0].set_xlabel('Count')
axes[0].set_title('Content Type Distribution', fontweight='bold', color='white')

# Language
lang = df['detected_language'].value_counts()
colors_lang = plt.cm.plasma(np.linspace(0.2, 0.85, len(lang)))
axes[1].barh(lang.index[::-1], lang.values[::-1], color=colors_lang[::-1], edgecolor='none')
for i, v in enumerate(lang.values[::-1]):
    axes[1].text(v + 2, i, str(v), va='center', fontsize=9, color='#ccc')
axes[1].set_xlabel('Count')
axes[1].set_title('Detected Language Distribution', fontweight='bold', color='white')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# is_short pie
short_counts = df['is_short'].value_counts()
axes[0].pie(short_counts, labels=['Regular', 'Short'], colors=['#3b82f6', '#f97316'],
            autopct='%1.1f%%', startangle=90, textprops={'color': 'white', 'fontsize': 11})
axes[0].set_title('Shorts vs Regular', fontweight='bold', color='white')

# has_hashtags pie
hash_counts = df['has_hashtags'].value_counts()
axes[1].pie(hash_counts, labels=['No Hashtags', 'Has Hashtags'], colors=['#a855f7', '#10b981'],
            autopct='%1.1f%%', startangle=90, textprops={'color': 'white', 'fontsize': 11})
axes[1].set_title('Hashtags Usage', fontweight='bold', color='white')

for ax in axes:
    ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 4. Top 50 Rankings

In [ ]:
top50 = df.head(50).iloc[::-1]

fig, ax = plt.subplots(figsize=(12, 16))
colors = plt.cm.hot(np.linspace(0.2, 0.85, 50))
bars = ax.barh(range(50), top50['views_num'].values / 1e9, color=colors, edgecolor='none', height=0.8)

ax.set_yticks(range(50))
ax.set_yticklabels([t[:55] + '...' if len(t) > 55 else t for t in top50['title'].values], fontsize=9)
ax.set_xlabel('Views (Billions)', fontsize=12, color='#e0e0e0')
ax.set_title('Top 50 Most Viewed YouTube Videos', fontsize=16, fontweight='bold', color='white', pad=15)

for bar, v in zip(bars, top50['views_num'].values):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2, f'{v/1e9:.1f}B',
            va='center', fontsize=8, color='#ccc')

ax.spines[['top', 'right']].set_visible(False)
ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 5. Views vs Likes Correlation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

x = df['views_num'] / 1e9
y = df['likes_num'] / 1e6

sc = ax.scatter(x, y, c=df['rank'], cmap='plasma', alpha=0.6, s=20, edgecolors='#333', linewidth=0.3)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Rank', color='#ccc')

z = np.polyfit(x, y, 1)
p = np.poly1d(z)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, p(x_line), 'w--', alpha=0.5, linewidth=2, label=f'Trend (slope={z[0]:.1f})')

corr = df['views_num'].corr(df['likes_num'])
ax.text(0.05, 0.95, f'Pearson r = {corr:.3f}', transform=ax.transAxes, fontsize=11,
        color='#f97316', fontweight='bold', va='top')

ax.set_xlabel('Views (Billions)', fontsize=12, color='#e0e0e0')
ax.set_ylabel('Likes (Millions)', fontsize=12, color='#e0e0e0')
ax.set_title('Views vs Likes Correlation', fontsize=14, fontweight='bold', color='white')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 6. Views Drop-off by Rank

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(df['rank'], df['views_num']/1e9, alpha=0.3, color='#3b82f6')
ax.plot(df['rank'], df['views_num']/1e9, color='#3b82f6', linewidth=2)

for r in [1, 50, 100, 250, 500, 750, 1000]:
    row = df[df['rank'] == r].iloc[0]
    ax.annotate(f'#{r}: {row["views_num"]/1e9:.1f}B',
                xy=(r, row['views_num']/1e9), xytext=(r+30, row['views_num']/1e9 + 0.3),
                fontsize=8, color='#f97316',
                arrowprops=dict(arrowstyle='->', color='#f97316', lw=0.8))

ax.set_xlabel('Rank', fontsize=12, color='#e0e0e0')
ax.set_ylabel('Views (Billions)', fontsize=12, color='#e0e0e0')
ax.set_title('Views Drop-off Across Rankings', fontsize=14, fontweight='bold', color='white')
ax.spines[['top', 'right']].set_visible(False)
ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 7. Content Type Analysis

In [ ]:
ct_stats = df.groupby('content_type').agg(
    count=('rank', 'size'),
    avg_views=('views_num', 'mean'),
    avg_likes=('likes_num', 'mean'),
    avg_engagement=('like_view_ratio', 'mean')
).sort_values('count', ascending=False)
ct_stats.round(2)

In [ ]:
top_ct = ct_stats.head(6)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#3b82f6', '#f97316', '#10b981', '#a855f7', '#ef4444', '#eab308']

# Avg views by content type
axes[0].barh(top_ct.index[::-1], top_ct['avg_views'][::-1] / 1e9, color=colors[::-1], edgecolor='none')
axes[0].set_xlabel('Avg Views (Billions)')
axes[0].set_title('Average Views by Content Type', fontweight='bold', color='white')

# Avg engagement by content type
axes[1].barh(top_ct.index[::-1], top_ct['avg_engagement'][::-1], color=colors[::-1], edgecolor='none')
axes[1].set_xlabel('Avg Like/View Ratio (%)')
axes[1].set_title('Avg Engagement by Content Type', fontweight='bold', color='white')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 8. Shorts vs Regular Analysis

In [ ]:
shorts = df[df['is_short'] == 1]
regular = df[df['is_short'] == 0]

print(f'Regular Videos: {len(regular)} | Avg Views: {regular["views_num"].mean()/1e9:.2f}B | Avg Likes: {regular["likes_num"].mean()/1e6:.1f}M')
print(f'Shorts:         {len(shorts)} | Avg Views: {shorts["views_num"].mean()/1e9:.2f}B | Avg Likes: {shorts["likes_num"].mean()/1e6:.1f}M')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Views comparison
axes[0].hist(regular['views_num']/1e9, bins=30, alpha=0.7, color='#3b82f6', label='Regular', edgecolor='#1a1a2e')
axes[0].hist(shorts['views_num']/1e9, bins=30, alpha=0.7, color='#f97316', label='Shorts', edgecolor='#1a1a2e')
axes[0].set_xlabel('Views (Billions)')
axes[0].set_ylabel('Count')
axes[0].set_title('Views: Shorts vs Regular', fontweight='bold', color='white')
axes[0].legend()

# Engagement comparison
axes[1].hist(regular['like_view_ratio'], bins=30, alpha=0.7, color='#3b82f6', label='Regular', edgecolor='#1a1a2e')
axes[1].hist(shorts['like_view_ratio'], bins=30, alpha=0.7, color='#f97316', label='Shorts', edgecolor='#1a1a2e')
axes[1].set_xlabel('Like/View Ratio (%)')
axes[1].set_ylabel('Count')
axes[1].set_title('Engagement: Shorts vs Regular', fontweight='bold', color='white')
axes[1].legend()

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 9. Top 50 by Engagement

In [ ]:
top_ratio = df.nlargest(50, 'like_view_ratio').iloc[::-1]

fig, ax = plt.subplots(figsize=(12, 14))
colors = plt.cm.cool(np.linspace(0.2, 0.85, 50))
bars = ax.barh(range(50), top_ratio['like_view_ratio'].values, color=colors, edgecolor='none', height=0.8)

ax.set_yticks(range(50))
ax.set_yticklabels([t[:55] + '...' if len(t) > 55 else t for t in top_ratio['title'].values], fontsize=9)
ax.set_xlabel('Likes / Views (%)', fontsize=12, color='#e0e0e0')
ax.set_title('Top 50 Videos by Like-to-View Ratio', fontsize=16, fontweight='bold', color='white', pad=15)

for bar, v in zip(bars, top_ratio['like_view_ratio'].values):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2, f'{v:.1f}%',
            va='center', fontsize=8, color='#ccc')

ax.spines[['top', 'right']].set_visible(False)
ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 10. Correlation Heatmap

In [ ]:
numeric_cols = ['rank', 'title_length', 'is_short', 'has_hashtags', 'views_num', 'likes_num', 'like_view_ratio']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8},
            annot_kws={'fontsize': 10})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', color='white')
ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 11. Key Insights

In [ ]:
print('=' * 60)
print('KEY INSIGHTS')
print('=' * 60)
print(f'\n1. #1 video "{df.iloc[0]["title"][:50]}..." has {df.iloc[0]["views_num"]/1e9:.1f}B views')
print(f'   That is {df.iloc[0]["views_num"] / df.iloc[1]["views_num"]:.1f}x the #2 video')
print(f'\n2. Most common content type: {df["content_type"].mode().values[0]} ({(df["content_type"]==df["content_type"].mode().values[0]).sum()} videos)')
print(f'\n3. Most common language: {df["detected_language"].mode().values[0]} ({(df["detected_language"]=="English").sum()} videos, {(df["detected_language"]=="English").sum()/len(df)*100:.1f}%)')
print(f'\n4. Shorts: {len(shorts)} ({len(shorts)/len(df)*100:.1f}% of top 1000)')
print(f'   Avg engagement: Shorts {shorts["like_view_ratio"].mean():.2f}% vs Regular {regular["like_view_ratio"].mean():.2f}%')
print(f'\n5. Views-Likes correlation: {corr.loc["views_num", "likes_num"]:.3f}')
print(f'\n6. Views range: {df["views_num"].min()/1e9:.2f}B (#{df.loc[df["views_num"].idxmin(), "rank"]:.0f}) to {df["views_num"].max()/1e9:.1f}B (#1)')
print(f'\n7. Hashtags: {df["has_hashtags"].sum()} videos ({df["has_hashtags"].sum()/len(df)*100:.1f}%) use hashtags')
print(f'\n8. Highest engagement: #{df.loc[df["like_view_ratio"].idxmax(), "rank"]:.0f} "{df.loc[df["like_view_ratio"].idxmax(), "title"][:40]}..." at {df["like_view_ratio"].max():.1f}%')